In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Leitura INMET com PySpark") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

caminho_cidades = "../data/bronze/coords_cidades.parquet"
caminho_clima = "../data/silver/INMET_PARQUET/Ano=2025"

df_cidades = spark.read.parquet(caminho_cidades)
df_clima = spark.read.parquet(caminho_clima)

print("Schema do DataFrame df_cidades:")
df_cidades.printSchema()

print("Schema do DataFrame df_clima:")
df_clima.printSchema()
#print("\nAlgumas linhas do DataFrame:")
#df.show()

df_cidades.createOrReplaceTempView("cidades")
df_clima.createOrReplaceTempView("clima")


# Para encerrar a sessão
#spark.stop()

Schema do DataFrame df_cidades:
root
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Cidade: string (nullable = true)
 |-- __index_level_0__: long (nullable = true)

Schema do DataFrame df_clima:
root
 |-- Data: string (nullable = true)
 |-- Tempo: string (nullable = true)
 |-- Temperatura: float (nullable = true)
 |-- Precipitacao: float (nullable = true)
 |-- Umidade: float (nullable = true)
 |-- Vento: float (nullable = true)
 |-- Regiao: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- Estacao: string (nullable = true)
 |-- Latitude: float (nullable = true)
 |-- Longitude: float (nullable = true)
 |-- Altitude: float (nullable = true)



In [10]:
df_resultado_sql = spark.sql("""
SELECT COUNT(CASE WHEN cidades.Latitude is null THEN 1 END) QtdVazio
        ,COUNT(CASE WHEN cidades.Latitude is not null THEN 1 END) QtdPreenchido
        ,COUNT(*) as QtdTotal
FROM clima
LEFT JOIN cidades
    ON CAST(cidades.Latitude as float) = clima.Latitude
    AND CAST(cidades.Longitude as float) = clima.Longitude
""")

print("Resultado Preenchimento dados de cidades:")
df_resultado_sql.show()

Resultado Preenchimento dados de cidades:


[Stage 15:======================================>                   (2 + 1) / 3]

+--------+-------------+--------+
|QtdVazio|QtdPreenchido|QtdTotal|
+--------+-------------+--------+
|  110079|      2277412| 2387491|
+--------+-------------+--------+

